# 04. Aircraft Generation Analysis — porównanie generacji samolotów

**Projekt:** PRC Data Challenge 2025  
**Dane:** `pk_big_data.curated.fuel_segments_train` — wyłącznie zbiór treningowy.

**Pytanie badawcze:** czy w porównywalnych warunkach obserwowane tempo spalania jest niższe dla nowszych generacji samolotów?

W notebooku **02** sprawdzono jakość danych, a w **03** tempo spalania poszczególnych typów. Tutaj badamy **generację konstrukcji**, nie rzeczywisty wiek egzemplarza. Nie powtarzamy EDA. Wskaźnik kg/h opisuje zaobserwowane odcinki lotu — **nie** efektywność na pasażerokilometr.

Wynik notebooka 03 dla A20N i A320 (~46,4% różnicy w warstwach) jest **punktem odniesienia, nie założeniem**. Wyniki notebooka 04 są liczone niezależnie na bieżących danych.

## 1. Importy, tabela i kontrola schematu

Dane źródłowe pozostają bez zmian; tworzymy wyłącznie robocze DataFrame'y.

In [0]:
import pyspark.sql.functions as F
from pyspark.sql import Window

CATALOG = "pk_big_data"
SCHEMA = "curated"
SOURCE_TABLE = f"{CATALOG}.{SCHEMA}.fuel_segments_train"

train = spark.table(SOURCE_TABLE)
aircraft_col = "flight_aircraft_type" if "flight_aircraft_type" in train.columns else "aircraft_type"
required = {"flight_id", "fuel_kg", "dur_s", "route_km", aircraft_col}
missing = required - set(train.columns)
assert not missing, f"Brak wymaganych kolumn: {sorted(missing)}"

print(f"Tabela: {SOURCE_TABLE}")
print(f"Typ samolotu: {aircraft_col}")
print(f"Liczba wierszy: {train.count():,}")

Tabela: pk_big_data.curated.fuel_segments_train
Typ samolotu: flight_aircraft_type
Liczba wierszy: 131,530


## 2. Generacja i porównywalne rodziny

Klasyfikacja robocza dotyczy **modeli konstrukcyjnych**:

| Rodzina porównawcza | Starsza generacja | Nowsza generacja |
|---|---|---|
| Airbus A320 | A320 | A20N |
| Airbus A321 | A321 | A21N |
| Boeing 737-800 / MAX 8 | B738 | B38M |

B39M pozostaje w szerszej grupie nowszych konstrukcji, ale nie jest bezpośrednim odpowiednikiem B738 w wybranych parach. Pozostałych typów nie zaliczamy arbitralnie do „starszych”.

In [0]:

# Typy używane do bezpośredniego porównania generacji
newer = ["A20N", "A21N", "B38M", "B39M"]
older = ["A320", "A321", "B738"]

# Rozszerzona klasyfikacja wszystkich typów
modern = ["A359", "B788", "B789"]

other = ["A388", "B748"]

older_other = [
    "A318", "A319", "B737", "B739",
    "A332", "A333", "B772", "B77W",
    "B77L", "B744", "B752", "B763",
    "A306", "MD11"
]

base = (
    train
    .withColumn(
        "aircraft_type_norm",
        F.upper(F.trim(F.col(aircraft_col)))
    )
    .withColumn(
        "generation",
        F.when(F.col("aircraft_type_norm").isin(newer), "nowsza")
         .when(F.col("aircraft_type_norm").isin(older + older_other), "starsza")
         .when(F.col("aircraft_type_norm").isin(modern), "nowoczesna_konstrukcja")
         .when(F.col("aircraft_type_norm").isin(other), "osobna_kategoria")
         .otherwise("nieznana")
    )
    .withColumn(
        "family",
        F.when(
            F.col("aircraft_type_norm").isin("A320", "A20N"),
            "A320 / A20N"
        )
        .when(
            F.col("aircraft_type_norm").isin("A321", "A21N"),
            "A321 / A21N"
        )
        .when(
            F.col("aircraft_type_norm").isin("B738", "B38M"),
            "B738 / B38M"
        )
        .otherwise("poza_parami")
    )
    .filter(F.col("fuel_kg").isNotNull() & (F.col("fuel_kg") >= 0))
    .filter(F.col("dur_s").between(300, 3600))
    .withColumn(
        "rate_kg_h",
        F.col("fuel_kg") / F.col("dur_s") * 3600.0
    )
)

display(
    base.groupBy("generation", "aircraft_type_norm")
    .agg(
        F.count("*").alias("n_segments"),
        F.countDistinct("flight_id").alias("n_flights")
    )
    .orderBy("generation", F.desc("n_segments"))
)


generation,aircraft_type_norm,n_segments,n_flights
nowoczesna_konstrukcja,A359,15005,1532
nowoczesna_konstrukcja,B788,7047,583
nowoczesna_konstrukcja,B789,1430,215
nowsza,A20N,25789,3169
nowsza,A21N,1426,361
nowsza,B38M,4,4
nowsza,B39M,1,1
osobna_kategoria,B748,14,3
osobna_kategoria,A388,6,3
starsza,A320,17405,2971


## 3. Opisowe porównanie generacji i kontrola liczebności

**Tempo ważone czasem** = suma `fuel_kg` / suma `dur_s` × 3600. Nie używamy średniej indywidualnych wartości kg/h, która nadmiernie waży krótkie przedziały. Wynik grupy „nowsza” lub „starsza” zależy od **składu typów**, więc jest wyłącznie opisowy.

In [0]:
classified = base.filter(F.col("generation") != "nieznana")

def summary_by(df, cols):
    return (df.groupBy(*cols)
        .agg(F.count("*").alias("n_segments"),
             F.countDistinct("flight_id").alias("n_flights"),
             F.sum("fuel_kg").alias("total_fuel_kg"),
             F.sum("dur_s").alias("total_duration_s"))
        .withColumn("rate_kg_h", F.col("total_fuel_kg") / F.col("total_duration_s") * 3600.0)
        .withColumn("co2_rate_kg_h", F.col("rate_kg_h") * 3.16))

print("Wynik zbiorczy — opisowy, nie jest efektem generacji:")
display(summary_by(classified, ["generation"]).orderBy("generation"))
print("Liczebność i tempo w poszczególnych rodzinach:")
family_df = classified.filter(F.col("family") != "poza_parami")
display(summary_by(family_df, ["family", "aircraft_type_norm", "generation"])
    .orderBy("family", "generation"))

Wynik zbiorczy — opisowy, nie jest efektem generacji:


generation,n_segments,n_flights,total_fuel_kg,total_duration_s,rate_kg_h,co2_rate_kg_h
nowoczesna_konstrukcja,23482,2330,3.589548595990402E7,2.3676615670000065E7,5457.864048508844,17246.850393287947
nowsza,27220,3535,5244108.756181616,1.4381492477999961E7,1312.7143480507038,4148.177339840225
osobna_kategoria,20,6,110616.8416,22219.451999999997,17922.162515979246,56634.033550494416
starsza,31952,4895,2.2246922549936775E7,2.0998260429000065E7,3814.074096783941,12052.474145837254


Liczebność i tempo w poszczególnych rodzinach:


family,aircraft_type_norm,generation,n_segments,n_flights,total_fuel_kg,total_duration_s,rate_kg_h,co2_rate_kg_h
A320 / A20N,A20N,nowsza,25789,3169,4802004.791165622,1.3574611719999967E7,1273.4962593977075,4024.248179696756
A320 / A20N,A320,starsza,17405,2971,6976480.556150399,1.0418867611999981E7,2410.5623506737656,7617.3770281291
A321 / A21N,A21N,nowsza,1426,361,439266.98555999977,803514.155,1968.0563661208919,6219.058116942018
A321 / A21N,A321,starsza,1157,274,534046.350148,802950.517,2394.3777603082362,7566.233722574027
B738 / B38M,B38M,nowsza,4,4,1400.0,2049.2380000000003,2459.450781217213,7771.864468646393
B738 / B38M,B738,starsza,5737,532,1873775.7990399979,2880639.618999997,2341.6996808804906,7399.770991582351


## 4. Pary w porównywalnych warunkach: czas i dystans

Stosujemy tę samą metodę co w notebooku 03: warstwy długości przedziału i dystansu `route_km` (odległość między lotniskami, **nie** droga przebyta w przedziale). Zatrzymujemy tylko warstwy, gdzie **oba** typy mają co najmniej 30 przedziałów i 5 różnych lotów. Wspólna waga warstwy to mniejsza liczba przedziałów z obu typów. Dzięki temu porównujemy pary przy takim samym rozkładzie wag warstw, ale nie kontrolujemy wszystkich warunków operacyjnych.

**Uwaga:** brak wystarczającej liczby wspólnych warstw to wynik analizy, nie powód do wymuszania porównania.

In [0]:
MIN_SEGMENTS = 30
MIN_FLIGHTS = 5

paired = (family_df
    .filter(F.col("route_km").isNotNull() & (F.col("route_km") >= 0))
    .withColumn("duration_bin", F.when(F.col("dur_s") < 600, "05-10 min")
                               .when(F.col("dur_s") < 1200, "10-20 min")
                               .when(F.col("dur_s") < 2400, "20-40 min")
                               .otherwise("40-60 min"))
    .withColumn("route_bin", F.when(F.col("route_km") < 1000, "<1000 km")
                            .when(F.col("route_km") < 3000, "1000-2999 km")
                            .when(F.col("route_km") < 6000, "3000-5999 km")
                            .otherwise("6000+ km")))

keys = ["family", "duration_bin", "route_bin"]
strata = summary_by(paired, keys + ["aircraft_type_norm", "generation"])
w = Window.partitionBy(*keys)
common = (strata
    .withColumn("types_present", F.count("*").over(w))
    .withColumn("min_segments", F.min("n_segments").over(w))
    .withColumn("min_flights", F.min("n_flights").over(w))
    .filter((F.col("types_present") == 2)
            & (F.col("min_segments") >= MIN_SEGMENTS)
            & (F.col("min_flights") >= MIN_FLIGHTS))
    .withColumn("shared_weight", F.col("min_segments")))

print("Wspólne warstwy spełniające progi:")
display(common.select(*keys, "aircraft_type_norm", "generation", "n_segments", "n_flights", "rate_kg_h", "shared_weight")
    .orderBy(*keys, "generation"))

matched = (common.groupBy("family", "generation")
    .agg(F.count("*").alias("n_common_strata"),
         F.sum(F.col("rate_kg_h") * F.col("shared_weight")).alias("weighted_sum"),
         F.sum("shared_weight").alias("weight_sum"))
    .withColumn("matched_rate_kg_h", F.col("weighted_sum") / F.col("weight_sum")))

print("Tempo porównawcze, ważone wspólnymi warstwami:")
display(matched.select("family", "generation", "n_common_strata", "matched_rate_kg_h")
    .orderBy("family", "generation"))

Wspólne warstwy spełniające progi:


family,duration_bin,route_bin,aircraft_type_norm,generation,n_segments,n_flights,rate_kg_h,shared_weight
A320 / A20N,05-10 min,1000-2999 km,A20N,nowsza,6495,1159,1264.22726454912,6495
A320 / A20N,05-10 min,1000-2999 km,A320,starsza,9348,1889,2376.96945129502,6495
A320 / A20N,05-10 min,3000-5999 km,A20N,nowsza,13863,1791,1197.4183049970743,2535
A320 / A20N,05-10 min,3000-5999 km,A320,starsza,2535,615,2670.0302409980854,2535
A320 / A20N,05-10 min,<1000 km,A20N,nowsza,110,36,1425.8628494450677,110
A320 / A20N,05-10 min,<1000 km,A320,starsza,245,96,2631.0297646847375,110
A320 / A20N,10-20 min,1000-2999 km,A20N,nowsza,1540,789,1732.7373109131202,1540
A320 / A20N,10-20 min,1000-2999 km,A320,starsza,2776,1348,2370.022337800711,1540
A320 / A20N,10-20 min,3000-5999 km,A20N,nowsza,1546,1011,1158.5600475646693,502
A320 / A20N,10-20 min,3000-5999 km,A320,starsza,502,277,2418.9344138283923,502


Tempo porównawcze, ważone wspólnymi warstwami:


family,generation,n_common_strata,matched_rate_kg_h
A320 / A20N,nowsza,10,1308.9093524677874
A320 / A20N,starsza,10,2442.907539431919
A321 / A21N,nowsza,5,1974.002407562554
A321 / A21N,starsza,5,2368.6554388256045


## 5. Różnica procentowa w obrębie każdej rodziny

`(tempo_nowszej / tempo_starszej − 1) × 100%`. Wartość ujemna oznacza niższe **obserwowane tempo spalania** nowszej generacji. Nie traktujemy jej jako przyczynowego efektu samej technologii.

Jeśli dana para ma za mało obserwacji, pozostawiamy ją bez wyniku procentowego.

In [0]:
older_rates = (matched.filter(F.col("generation") == "starsza")
    .select("family", F.col("matched_rate_kg_h").alias("older_rate"),
            F.col("n_common_strata").alias("older_strata")))
newer_rates = (matched.filter(F.col("generation") == "nowsza")
    .select("family", F.col("matched_rate_kg_h").alias("newer_rate"),
            F.col("n_common_strata").alias("newer_strata")))

all_families = spark.createDataFrame(
    [("A320 / A20N",), ("A321 / A21N",), ("B738 / B38M",)], ["family"]
)
pair_results = (all_families
    .join(older_rates, "family", "left")
    .join(newer_rates, "family", "left")
    .withColumn("diff_pct", (F.col("newer_rate") / F.col("older_rate") - 1.0) * 100.0)
    .withColumn("status", F.when(F.col("older_rate").isNull() | F.col("newer_rate").isNull(),
                                 "brak dostatecznych wspólnych warstw")
                           .otherwise("porównanie dostępne")))

display(pair_results.orderBy("family"))

family,older_rate,older_strata,newer_rate,newer_strata,diff_pct,status
A320 / A20N,2442.907539431917,10,1308.9093524677883,10,-46.420020760500535,porównanie dostępne
A321 / A21N,2368.655438825605,5,1974.0024075625558,5,-16.661479115709653,porównanie dostępne
B738 / B38M,null,null,null,null,null,brak dostatecznych wspólnych warstw


## 6. Analiza wrażliwości

Dla każdej pary zestawiamy tempo ważone czasem dla wszystkich dodatnich przedziałów, przedziałów 5–60 min i przedziałów 5–60 min bez globalnego górnego 1% indywidualnych wartości kg/h. **Wykluczenie skrajnych temp dotyczy tylko analizy wrażliwości; nie usuwa danych źródłowych.**

In [0]:
all_positive = (train
    .withColumn("aircraft_type_norm", F.upper(F.trim(F.col(aircraft_col))))
    .filter(F.col("aircraft_type_norm").isin(older + newer))
    .filter(F.col("fuel_kg").isNotNull() & (F.col("fuel_kg") >= 0))
    .filter(F.col("dur_s").isNotNull() & (F.col("dur_s") > 0))
    .withColumn("rate_kg_h", F.col("fuel_kg") / F.col("dur_s") * 3600.0))

p99_values = classified.approxQuantile("rate_kg_h", [0.99], 0.001)
p99 = p99_values[0] if p99_values else None

variants = [
    ("wszystkie dodatnie", all_positive),
    ("5-60 min", classified),
]
if p99 is not None:
    variants.append(("5-60 min bez top 1%", classified.filter(F.col("rate_kg_h") <= p99)))

sensitivity = None
for name, subset in variants:
    item = (summary_by(subset, ["aircraft_type_norm"])
        .withColumn("variant", F.lit(name))
        .select("variant", "aircraft_type_norm", "n_segments", "n_flights", "rate_kg_h"))
    sensitivity = item if sensitivity is None else sensitivity.unionByName(item)

print("Globalny próg p99 kg/h dla 5–60 min:", p99)
display(sensitivity.orderBy("aircraft_type_norm", "variant"))

Globalny próg p99 kg/h dla 5–60 min: 10757.900748404578


variant,aircraft_type_norm,n_segments,n_flights,rate_kg_h
5-60 min,A20N,25789,3169,1273.4962593977007
5-60 min bez top 1%,A20N,25789,3169,1273.4962593977007
wszystkie dodatnie,A20N,40488,3220,1270.7041629079004
5-60 min,A21N,1426,361,1968.0563661208935
5-60 min bez top 1%,A21N,1426,361,1968.0563661208935
wszystkie dodatnie,A21N,3138,393,2010.646678946212
5-60 min,A306,44,9,8799.333923910599
5-60 min bez top 1%,A306,35,9,7209.049013132894
5-60 min,A318,4,1,848.1389136431582
5-60 min bez top 1%,A318,4,1,848.1389136431582


## 7. Szacowana emisja CO₂

Zależność `co2_kg = fuel_kg × 3,16` jest czysto arytmetyczna. **Procentowa różnica emisji CO₂/h jest identyczna jak różnica kg paliwa/h**. Nie jest to niezależny dowód przewagi którejkolwiek generacji.

In [0]:
emissions = (pair_results
    .withColumn("older_co2_kg_h", F.col("older_rate") * 3.16)
    .withColumn("newer_co2_kg_h", F.col("newer_rate") * 3.16)
    .select("family", "older_co2_kg_h", "newer_co2_kg_h", "diff_pct", "status"))
display(emissions.orderBy("family"))

family,older_co2_kg_h,newer_co2_kg_h,diff_pct,status
A320 / A20N,7719.587824604858,4136.153553798211,-46.420020760500535,porównanie dostępne
A321 / A21N,7484.9511866889125,6237.847607897676,-16.661479115709653,porównanie dostępne
B738 / B38M,null,null,null,brak dostatecznych wspólnych warstw



## 8. Wnioski z analizy generacji samolotów

### 8.1. Klasyfikacja generacji samolotów

W analizie uwzględniono wszystkie 26 typów samolotów występujących w zbiorze treningowym. Podzielono je na cztery grupy: starsze konstrukcje, nowsze generacje, nowoczesne konstrukcje oraz samoloty należące do osobnej kategorii.

Po ograniczeniu danych do przedziałów trwających 5–60 minut otrzymano 82 674 obserwacje.

| Grupa | Liczba przedziałów | Tempo zużycia paliwa [kg/h] |
|---|---:|---:|
| Nowsza generacja | 27 220 | 1312,71 |
| Starsza konstrukcja | 31 952 | 3814,07 |
| Nowoczesna konstrukcja | 23 482 | 5457,86 |
| Osobna kategoria | 20 | 17 922,16 |

Różnice między grupami są znaczne, jednak wyników zbiorczych nie można interpretować jako bezpośredniego wpływu generacji na zużycie paliwa.

Grupy obejmują samoloty o różnej wielkości, przeznaczeniu i charakterystyce operacyjnej. Dlatego główne porównania przeprowadzono w obrębie podobnych rodzin konstrukcyjnych.

### 8.2. Porównanie starszych i nowszych generacji

W celu ograniczenia wpływu różnic operacyjnych porównano typy samolotów w podobnych zakresach czasu trwania przedziałów oraz dystansu między lotniskami.

W analizie uwzględniono wyłącznie wspólne warstwy spełniające minimalne wymagania dotyczące liczby przedziałów i lotów.

| Porównanie | Starsza generacja [kg/h] | Nowsza generacja [kg/h] | Różnica |
|---|---:|---:|---:|
| A320 vs A20N | 2442,91 | 1308,91 | -46,42% |
| A321 vs A21N | 2368,66 | 1974,00 | -16,66% |
| B738 vs B38M | — | — | Niewystarczająca próba |

**W obu możliwych do przeprowadzenia porównaniach nowsze generacje Airbusów wykazują niższe obserwowane tempo zużycia paliwa.**

W przypadku A20N różnica wynosi około 46,42%, natomiast dla A21N około 16,66%.

Wyniki wskazują na związek pomiędzy generacją konstrukcji a tempem spalania, jednak wielkość tej różnicy nie jest jednakowa dla wszystkich rodzin.

### 8.3. Znaczenie porównywalnych warunków lotu

Porównanie przeprowadzono w 10 wspólnych warstwach dla A320/A20N oraz 5 dla A321/A21N.

Warstwy uwzględniały:
- Czas trwania analizowanego przedziału.
- Dystans geograficzny między lotniskami wylotu i przylotu.
- Minimalną liczebność obserwacji i lotów w obu grupach.

Dzięki temu częściowo ograniczono wpływ różnic w strukturze analizowanych lotów.

Uzyskane różnice utrzymują się również po takim dopasowaniu, co wzmacnia wiarygodność zaobserwowanych zależności.

Nie jest to jednak pełna kontrola warunków operacyjnych, ponieważ nie uwzględniono m.in. masy samolotu, liczby pasażerów, warunków pogodowych oraz rzeczywiście pokonanego dystansu.

### 8.4. Ograniczenia porównania Boeing 737

Nie udało się przeprowadzić wiarygodnego porównania B738 i B38M.

Po zastosowaniu ograniczenia długości przedziałów do 5–60 minut dostępne były:

- B738: 5737 przedziałów z 532 lotów.
- B38M: 4 przedziały z 4 lotów.
- B39M: 1 przedział z 1 lotu.

Tak mała liczba obserwacji nowszych wariantów uniemożliwia uzyskanie wystarczającej liczby wspólnych warstw porównawczych.

**Brak wyniku dla Boeing 737 MAX nie oznacza braku różnic w zużyciu paliwa.** Oznacza jedynie, że dostępne dane są niewystarczające do ich wiarygodnego oszacowania.

### 8.5. Nowoczesne samoloty szerokokadłubowe

W zbiorze występują również licznie reprezentowane nowoczesne konstrukcje szerokokadłubowe:

| Typ samolotu | Liczba przedziałów | Tempo zużycia paliwa [kg/h] |
|---|---:|---:|
| A359 | 15 005 | 5728,59 |
| B788 | 7047 | 4562,46 |
| B789 | 1430 | 5460,21 |

Samoloty te mają wyższe bezwzględne tempo spalania niż analizowane samoloty wąskokadłubowe.

Nie świadczy to jednak o ich niższej efektywności transportowej, ponieważ mogą przewozić większą liczbę pasażerów i ładunków na dłuższych trasach.

Dlatego nowoczesne samoloty szerokokadłubowe potraktowano jako odrębną grupę opisową, a nie bezpośrednich odpowiedników A320 i A321.

### 8.6. Stabilność wyników

Analiza wrażliwości obejmowała porównanie wszystkich dodatnich przedziałów, przedziałów trwających 5–60 minut oraz wariantu bez najwyższych indywidualnych wartości tempa spalania.

Dla najważniejszych porównywanych typów otrzymano:

| Typ | 5–60 min [kg/h] | Po ograniczeniu wartości skrajnych [kg/h] |
|---|---:|---:|
| A20N | 1273,50 | 1273,50 |
| A320 | 2410,56 | 2407,17 |
| A21N | 1968,06 | 1968,06 |
| A321 | 2394,38 | 2385,23 |

Wyniki pozostają stosunkowo stabilne po ograniczeniu wpływu skrajnych obserwacji.

Oznacza to, że podstawowe różnice między analizowanymi generacjami Airbusów nie wynikają wyłącznie z pojedynczych pomiarów o ekstremalnym tempie spalania.

### 8.7. Szacowana emisja CO₂

Na podstawie zużycia paliwa obliczono szacowaną emisję CO₂ z wykorzystaniem współczynnika 3,16 kg CO₂ na 1 kg paliwa.

| Porównanie | Starsza generacja [kg CO₂/h] | Nowsza generacja [kg CO₂/h] |
|---|---:|---:|
| A320 vs A20N | 7719,59 | 4136,15 |
| A321 vs A21N | 7484,95 | 6237,85 |

W analizowanych warunkach nowsze generacje wykazują również niższą szacowaną emisję CO₂ na godzinę obserwowanego lotu.

Różnica procentowa emisji jest jednak identyczna z różnicą tempa zużycia paliwa, ponieważ emisja została obliczona bezpośrednio z `fuel_kg`.

Nie stanowi więc niezależnego potwierdzenia efektywności środowiskowej.

### 8.8. Wniosek końcowy

Analiza wykazała, że **nowsze generacje samolotów Airbus A320neo i A321neo charakteryzują się niższym obserwowanym tempem zużycia paliwa niż odpowiadające im starsze warianty A320 i A321**.

Największą różnicę zaobserwowano dla A20N względem A320 (46,42%), natomiast dla A21N względem A321 wyniosła ona 16,66%.

Wyniki utrzymują się po częściowym uwzględnieniu długości przedziałów, dystansu trasy oraz ograniczeniu wpływu wartości skrajnych.

Jednocześnie analiza wykazała, że nie można stosować jednego współczynnika poprawy efektywności dla wszystkich nowszych generacji. Wielkość obserwowanych różnic zależy od porównywanej rodziny samolotów i struktury dostępnych danych.

**Główny wniosek:** wyniki są zgodne z hipotezą, że nowsze generacje samolotów mogą charakteryzować się niższym tempem zużycia paliwa, ale nie pozwalają jednoznacznie określić przyczynowego wpływu samej generacji konstrukcji.

Kolejnym etapem projektu będzie budowa modeli predykcyjnych, które pozwolą uwzględnić jednocześnie typ samolotu, parametry lotu, czas trwania przedziału oraz charakterystykę trasy.
